# Trajectory match: new PatternSearchCV in Run A's exact configuration

Run A (the 26-value-grid, R2-scored run recorded in the prototype notebook) vs
the new package configured identically: same grid AND same dimension order,
`scoring='r2'`, `data_zones=1` (every evaluation on 100% of rows), single start
at the grid midpoint. Every visited point is compared 1:1 against Run A's 18
recorded rows. Expected divergences and their causes: (a) initial step on the
26-value axis (prototype used ceil(len/2)=13, package uses (len-1)//2=12 — so
probe #2 is 250 vs Run A's 260); (b) the two deliberate bug fixes (failed
pattern moves no longer contract the mesh; pattern references compound).

In [ ]:
# --- Data pipeline: byte-for-byte replication of the prototype notebook ---
import time
import numpy as np
import pandas as pd

t0 = time.time()
trainBench = pd.read_csv(r"C:\FILES\Code\Benchmarking\train.csv")  # notebook: /home/dell/train.csv

SplitPoint = int(len(trainBench.index) * 0.8)
print("SplitPoint: ", SplitPoint)

df = trainBench

# int64 -> int32; object -> category -> numeric codes (Date included, as in the prototype)
Int64columns = df.select_dtypes(["int64"]).columns
df[Int64columns] = df[Int64columns].astype(np.int32)
cat_columns = df.select_dtypes(["object"]).columns
df[cat_columns] = df[cat_columns].astype("category")
df[cat_columns] = df[cat_columns].apply(lambda x: x.cat.codes)

# the five weather columns the prototype drops (note the dataset's own 'kM' typo)
df = df.drop("Max_Gust_SpeedKm_h", axis=1)
df = df.drop("CloudCover", axis=1)
df = df.drop("Max_VisibilityKm", axis=1)
df = df.drop("Min_VisibilitykM", axis=1)
df = df.drop("Mean_VisibilityKm", axis=1)

# positional 80/20 chronological split
trainBench, validBench = df.iloc[:SplitPoint, :], df.iloc[SplitPoint:, :]
print("Training Set shape", trainBench.shape)
print("Validation Set shape", validBench.shape)
del df

# feature mask: everything but the target (alphabetical order, incl. Date codes
# and NumberOfCustomers, exactly as the prototype had it)
mask = trainBench.columns.difference(["NumberOfSales"])
trainDataset_X = trainBench[mask]
trainDataset_y = trainBench["NumberOfSales"]
validBench_X = validBench[mask]
validBench_y = validBench["NumberOfSales"]
print("Feature Columns:")
print(list(mask))
del trainBench, validBench
print(f"Pipeline done in {time.time() - t0:.1f} s")

In [ ]:
# --- New PatternSearchCV in Run A's exact configuration ---
# 26-value grid, SAME dimension order as Run A's param_dist, R^2 scoring,
# data_zones=1 (all evaluations on 100% of rows, like Run A), single start.
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import TimeSeriesSplit
from bayes_halving_search_cv import PatternSearchCV

X, y = trainDataset_X, trainDataset_y
clf = ExtraTreesRegressor(n_estimators=240, max_features=max(1, X.shape[1] - 15),
                          max_depth=16, n_jobs=1, random_state=0)

param_grid = {
    "max_features": [2, 3, 4],                                # Run A order
    "n_estimators": list(range(10, 261, 10)),                 # 26 values
    "max_depth": [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17],
}

search = PatternSearchCV(clf, param_grid, scoring="r2",
                         cv=TimeSeriesSplit(n_splits=5), n_jobs=-1,
                         data_zones=1, random_state=0, verbose=1)
start = time.time()
search.fit(X, y)
elapsed = time.time() - start
print()
print(f"New PatternSearchCV (Run A config) took {elapsed:.2f} s")


In [ ]:
# --- Point-by-point comparison against Run A's recorded trajectory ---
RUN_A = [  # (max_features, n_estimators, max_depth, R2) — from the original notebook
    (3, 130, 11, 0.671751), (4, 130, 11, 0.721118), (4, 260, 11, 0.715665),
    (4, 10, 11, 0.686570), (4, 130, 17, 0.809692), (3, 130, 17, 0.766341),
    (4, 200, 17, 0.805551), (4, 60, 17, 0.802977), (4, 130, 14, 0.774317),
    (4, 170, 17, 0.808907), (4, 90, 17, 0.806124), (4, 130, 15, 0.785468),
    (4, 150, 17, 0.809981), (4, 150, 16, 0.798049), (4, 50, 17, 0.803193),
    (3, 150, 17, 0.767527), (4, 160, 17, 0.808689), (4, 140, 17, 0.809143),
]
res = search.cv_results_
ours = [(p["max_features"], p["n_estimators"], p["max_depth"], s)
        for p, s in zip(res["params"], res["mean_test_score"])]

a_pts = {t[:3]: t[3] for t in RUN_A}
o_pts = {t[:3]: t[3] for t in ours}
shared = sorted(set(a_pts) & set(o_pts))

print(f"Run A evaluations : {len(RUN_A)}   (867.34 s, original machine)")
print(f"New   evaluations : {len(ours)}   ({elapsed:.2f} s, this machine)")
print(f"Shared grid points: {len(shared)} "
      f"({len(shared)/len(RUN_A)*100:.0f}% of Run A's trajectory)")
print()
print("side-by-side sequence (order of evaluation):")
print(f"{'#':>3s}  {'Run A (mf, n_est, depth)':30s}{'R2':>9s}   "
      f"{'New (mf, n_est, depth)':30s}{'R2':>9s}")
for i in range(max(len(RUN_A), len(ours))):
    a = RUN_A[i] if i < len(RUN_A) else None
    o = ours[i] if i < len(ours) else None
    a_s = f"{str(a[:3]):30s}{a[3]:>9.6f}" if a else " " * 39
    o_s = f"{str(o[:3]):30s}{o[3]:>9.6f}" if o else ""
    mark = "  <-- same point" if a and o and a[:3] == o[:3] else ""
    print(f"{i:>3d}  {a_s}   {o_s}{mark}")
print()
print("score agreement on shared points (determinism check across machines):")
for pt in shared:
    print(f"  {str(pt):18s} Run A {a_pts[pt]:.6f}  |  new {o_pts[pt]:.6f}"
          f"  |  diff {abs(a_pts[pt]-o_pts[pt]):.2e}")
print()
print(f"Run A best : (4, 150, 17)  R2 0.809981")
best = search.best_params_
print(f"New best   : ({best['max_features']}, {best['n_estimators']}, "
      f"{best['max_depth']})  R2 {search.best_score_:.6f}")
